# SuppleScope claim verifier: training on a free GPU

Runtime > Change runtime type > T4 GPU (Colab) or Accelerator GPU T4 (Kaggle).
Trains roberta-base and PubMedBERT on SciFact + HealthVer + PUBHEALTH (+ your SuppleScope gold labels), then zips both models for `VERIFIER_MODEL_DIRS`. No API key is needed.

In [ ]:
!git clone -b feature/complete-stack https://github.com/hasanalbannahorigin-nsu/SuppleScope.git
%cd SuppleScope/backend
!pip -q install -r requirements.txt
!pip -q install "transformers==4.46.3" "datasets==3.1.0" "accelerate==1.1.1" sentencepiece

Optional: upload your SuppleScope gold file (`verifier_gold.jsonl`, from labelling/import_annotations.py) to `backend/eval/`.

In [ ]:
import os
GOLD = 'eval/verifier_gold.jsonl'
extra = f'--supplescope {GOLD}' if os.path.exists(GOLD) else ''
!python -m verifier_training.prepare_data --out data/verifier --scifact --healthver --pubhealth $extra

In [ ]:
!python -m verifier_training.train --model roberta-base --data data/verifier --out models/verifier-roberta

In [ ]:
!python -m verifier_training.train --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --data data/verifier --out models/verifier-pubmedbert

In [ ]:
!rm -rf models/*/checkpoints
!cd models && zip -qr ../verifier_models.zip verifier-roberta verifier-pubmedbert
print('Download backend/verifier_models.zip, unzip into D:/CSE499/SuppleScope/backend/models/ and set\n'
      'VERIFIER_MODEL_DIRS=models/verifier-roberta,models/verifier-pubmedbert in .env')

If you have gold labels, compare every verifier (ablation):

In [ ]:
import os
os.environ['VERIFIER_MODEL_DIRS'] = 'models/verifier-roberta,models/verifier-pubmedbert'
if os.path.exists(GOLD):
    !python -m verifier_training.evaluate --gold $GOLD --modes rules,ensemble,ensemble+rules